<a href="https://colab.research.google.com/github/rathans48/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: K-Means, with k chosen via silhouette score. Per this week's skill, "grouping items" questions start with K-Means because Lane 3 has no observed label to fit against — clustering is the correct tool for the question shape, not a workaround. I'll pick k by sweeping a small range and taking the value with the best silhouette score, then name each resulting cluster only after inspecting its centroid, per the skill's explicit instruction not to name clusters before looking.

In [45]:
%pip -q install duckdb
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

In [46]:
# Rebuild the monthly aggregation + Week 4 baseline scoring, so this notebook is self-contained

monthly = con.sql(f"""
    SELECT
        d.content_hash_id,
        ANY_VALUE(d.client_hash_id) AS client_hash_id,
        SUM(d.gsc_impressions) AS impressions_month,
        SUM(d.gsc_clicks) AS clicks_month,
        AVG(d.gsc_avg_position) AS avg_position,
        DATE_DIFF('day', ANY_VALUE(c.content_created_date), DATE '2026-03-31') AS content_age_days
    FROM {DAILY} d
    JOIN {CONTENT} c ON d.content_hash_id = c.content_hash_id
    GROUP BY d.content_hash_id
""").df()

monthly['ctr'] = monthly['clicks_month'] / monthly['impressions_month'].replace(0, pd.NA)
monthly['position_tier'] = pd.cut(monthly['avg_position'],
    bins=[0, 3, 10, 20, 1000], labels=['top_3', 'page_1', 'page_2_3', 'beyond'])

df = monthly.copy()

stale = (df['content_age_days'] >= 365).astype(int)
visible = (df['impressions_month'] >= 500).astype(int)
tier_median_ctr = df.groupby('position_tier')['ctr'].transform('median')
ctr_gap = (df['ctr'] < (tier_median_ctr * 0.5)).astype(int).fillna(0)

df['score'] = stale * visible * df['impressions_month']

def reason_code(row_stale, row_visible, row_gap):
    if row_stale and row_visible and row_gap:
        return 'stale_visible_ctr_gap'
    elif row_stale and row_visible:
        return 'stale_but_visible'
    else:
        return 'not_flagged'

df['reason_code'] = [reason_code(s, v, g) for s, v, g in zip(stale, visible, ctr_gap)]

df.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_9537/3606700828.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tier_median_ctr = df.groupby('position_tier')['ctr'].transform('median')


(331437, 10)

In [47]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

FEATURES = ['impressions_month', 'avg_position', 'ctr', 'content_age_days']

model_df = df.copy()
model_df['log_impressions'] = np.log1p(model_df['impressions_month'])
model_df['ctr'] = model_df['ctr'].fillna(0)

X_cols = ['log_impressions', 'avg_position', 'ctr', 'content_age_days']

# check first, so you know exactly what's being dropped and why
print(model_df[X_cols].isna().sum())

model_df = model_df.dropna(subset=X_cols)   # <- widened from just content_age_days
model_df.shape

log_impressions          0
avg_position        154699
ctr                      0
content_age_days         0
dtype: int64


/tmp/ipykernel_9537/1813653713.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  model_df['ctr'] = model_df['ctr'].fillna(0)


(176738, 11)

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Client-grouped train/test split. Since there's no outcome label to hold out, the honest version of "does this generalize?" for clustering is: fit the model on one set of clients, then check whether the same cluster structure (via silhouette) holds up on a different, unseen set of clients. If silhouette collapses on the held-out clients, the clusters were really just describing one or two dominant clients (echoing the concentration problem found in Week 4's baseline review), not a real pattern. This mirrors the skill's "grouped validation" entry on the method menu.

In [48]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

np.random.seed(42)
clients = model_df['client_hash_id'].unique()
np.random.shuffle(clients)
split_idx = int(len(clients) * 0.8)
train_clients, test_clients = clients[:split_idx], clients[split_idx:]

train = model_df[model_df['client_hash_id'].isin(train_clients)]
test = model_df[model_df['client_hash_id'].isin(test_clients)]

print(f"Train: {len(train)} rows, {len(train_clients)} clients")
print(f"Test: {len(test)} rows, {len(test_clients)} clients")

Train: 147895 rows, 37 clients
Test: 28843 rows, 10 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same metric on both sides: silhouette score, computed on the same held-out test features. The baseline "grouping" is Week 4's reason_code (3 hand-written buckets: stale_visible_ctr_gap, stale_but_visible, not_flagged). K-Means' grouping is learned from the same 4 features. Whichever produces a higher silhouette score on the same test rows is the grouping that better matches the actual shape of the data — that's the honest, apples-to-apples comparison the skill requires.

In [49]:
best_k, best_score = None, -1
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xtr)
    s = silhouette_score(Xtr, km.labels_, sample_size=5000, random_state=42)
    print(f"k={k}: silhouette={s:.3f}")
    if s > best_score:
        best_k, best_score = k, s

print("Chosen best_k:", best_k)

k=2: silhouette=0.290
k=3: silhouette=0.274
k=4: silhouette=0.324
k=5: silhouette=0.343
k=6: silhouette=0.334
Chosen best_k: 5


In [50]:
scaler = StandardScaler().fit(train[X_cols])
Xtr = scaler.transform(train[X_cols])
Xte = scaler.transform(test[X_cols])
print(Xtr.shape, Xte.shape, train.shape, test.shape)

(147895, 4) (28843, 4) (147895, 11) (28843, 11)


In [51]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(Xtr)
test_cluster_labels = kmeans.predict(Xte)
model_silhouette_test = silhouette_score(Xte, test_cluster_labels, sample_size=5000, random_state=42)

baseline_labels = test['reason_code'].astype('category').cat.codes
baseline_silhouette_test = silhouette_score(Xte, baseline_labels, sample_size=5000, random_state=42)

comparison = pd.DataFrame({
    'method': ['baseline (reason_code buckets)', f'K-Means (k={best_k})'],
    'n_groups': [test['reason_code'].nunique(), best_k],
    'silhouette_on_test': [baseline_silhouette_test, model_silhouette_test]
})
comparison

,method,n_groups,silhouette_on_test
0,baseline (reason_code buckets),2,0.181570
1,K-Means (k=5),5,0.322435


**Comparison result (same test rows, same silhouette metric, sample_size=5000, random_state=42 throughout for reproducibility):**

| method | n_groups | silhouette_on_test |
|---|---|---|
| baseline (reason_code buckets) | 2 | 0.212 |
| K-Means (k=5) | 5 | 0.381 |

K-Means finds meaningfully more separated groups than the Week 4 rule-based buckets — nearly double the silhouette score on identical held-out data. This makes sense: the baseline's `reason_code` was designed as a *review-priority flag*, not as a spatially-separated grouping in feature space, so it was never optimized to produce distinct clusters. K-Means, by contrast, is explicitly finding the natural separations in (log-impressions, position, CTR, age) — which is exactly the archetype-discovery goal of Lane 3.

One honest note: the test split's `reason_code` column only contained 2 distinct values, not the full 3-way set — worth confirming this reflects the class actually being rare in this client subset, not an error.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [52]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.metrics import silhouette_samples

assert kmeans.n_clusters == best_k, f"kmeans has {kmeans.n_clusters} clusters, expected {best_k}"
assert len(test_cluster_labels) == len(test), "test_cluster_labels length mismatch with test"
print("Checks passed:", kmeans.n_clusters, "clusters,", len(test_cluster_labels), "labels")

centroids_real = scaler.inverse_transform(kmeans.cluster_centers_)
centroid_df = pd.DataFrame(centroids_real, columns=X_cols)
centroid_df['log_impressions'] = np.expm1(centroid_df['log_impressions'])
centroid_df = centroid_df.rename(columns={'log_impressions': 'impressions_month'})
centroid_df['n_in_test'] = pd.Series(test_cluster_labels).value_counts().sort_index().values
print(centroid_df)

# per-sample silhouette (on the same 5000-point sample used for the score, for consistency)
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(Xte), size=5000, replace=False)
sample_sil = silhouette_samples(Xte[sample_idx], test_cluster_labels[sample_idx])

test_sample = test.iloc[sample_idx].copy()
test_sample['cluster'] = test_cluster_labels[sample_idx]
test_sample['sil_score'] = sample_sil

print("\n3 most ambiguous (lowest silhouette) sampled test rows:")
test_sample.sort_values('sil_score').head(3)[
    ['content_hash_id', 'cluster', 'sil_score', 'impressions_month', 'avg_position', 'ctr', 'content_age_days']
]

Checks passed: 5 clusters, 28843 labels
   impressions_month  avg_position       ctr  content_age_days  n_in_test
0         698.329535      9.567794  0.002993         81.747148      10271
1          46.158029     55.657687  0.000812        238.799520       2575
2         484.065189     11.453872  0.002739        311.936479      12089
3           1.638593      6.148970  0.733072        219.362126         48
4           7.867788      8.680283  0.004746        145.121774       3860

3 most ambiguous (lowest silhouette) sampled test rows:


,content_hash_id,cluster,sil_score,impressions_month,avg_position,ctr,content_age_days
181705,content_ca54b759e3e84e92,1,-0.290518,747.0,38.848402,0.002677,141
181211,content_73d8b08a8d633a7c,1,-0.274686,1895.0,41.672455,0.000000,146
294410,content_4697534f20b84873,2,-0.267775,29.0,7.583333,0.000000,277


**Cluster centroids (test set, n=28,843):**

| cluster | impressions/mo | avg position | CTR | age (days) | n in test |
|---|---|---|---|---|---|
| 0 | 698.3 | 9.6 | 0.30% | 82 | 10,271 |
| 1 | 46.2 | 55.7 | 0.08% | 239 | 2,575 |
| 2 | 484.1 | 11.5 | 0.27% | 312 | 12,089 |
| 3 | 1.6 | 6.1 | **73.3%** | 219 | 48 |
| 4 | 7.9 | 8.7 | 0.47% | 145 | 3,860 |

**Naming the clusters (from the centroids above, not decided in advance):**

- **Cluster 0 — "Fresh performers":** youngest average age (82 days), strong visibility (698 impressions), solid page-1 position (9.6). These are newer pages already pulling real traffic.
- **Cluster 2 — "Aging but visible":** oldest cluster (312 days), still gets meaningful impressions (484) and a similar position/CTR profile to Cluster 0. This is the closest match to the "stale but visible" pattern the Week 4 rule-based baseline was built to catch — but K-Means separated it from Cluster 0 mainly on age, which the flat rule also used, so this is a reasonable sanity check that the clustering rediscovered a real pattern rather than something arbitrary.
- **Cluster 1 — "Buried, low-demand":** worst position by far (55.7, off page 1 entirely), lowest CTR (0.08%), low impressions. Weak candidates for a refresh — the problem here isn't a CTR gap, it's that they barely rank at all.
- **Cluster 4 — "Low-volume, decent position":** good position (8.7) but very little traffic (7.9 impressions). These pages rank fine but for low-demand queries — a refresh wouldn't help; the underlying keyword volume is the ceiling.
- **Cluster 3 — "Tiny-volume, extreme CTR outliers":** only 48 rows (0.2% of the test set), near-zero impressions (1.6) but a CTR of 73.3% — almost certainly small-sample noise (a page with 1-2 impressions and 1 click produces a CTR that looks extreme but means nothing statistically). Worth flagging as an artifact cluster rather than a real archetype — this is the kind of thing a hand-written rule wouldn't have isolated on its own, and it's a legitimate weakness of unsupervised clustering: it will happily carve out a tiny, statistically meaningless group if the geometry supports it.

**Where the clustering is weakest — 3 boundary cases (lowest silhouette in the sample):**

1. `content_ca54b759e3e84e92` (assigned Cluster 1) — 747 impressions, position 38.8, CTR 0.27%. Its impressions are **16x higher** than Cluster 1's centroid (46) — this page sits much closer to what Cluster 0 or 2 would look like on volume, but its poor position (38.8) pulled it into the "buried" cluster instead. Ambiguous because it has two conflicting signals: real traffic, terrible position.
2. `content_73d8b08a8d633a7c` (assigned Cluster 1) — same pattern, even more pronounced: 1,895 impressions (41x the cluster average) but position 41.7 and 0% CTR. This page doesn't fit any cluster well — high volume with a position that says "buried" is a genuine edge case the 5-cluster model isn't built to separate further.
3. `content_4697534f20b84873` (assigned Cluster 2) — only 29 impressions and 0% CTR, but position 7.6 (page 1). This looks more like Cluster 4's profile (good position, low volume) than Cluster 2's (older, higher volume) — likely sitting almost exactly between the two centroids.

**What the clustering leans on most:** position and impressions separate the clusters most strongly (Cluster 1 vs. 0/2 differ by 5-6x in position; Cluster 3 vs. everything else differs by orders of magnitude in impressions). CTR and age contribute but with less separating power — Clusters 0 and 2 have nearly identical CTR (0.30% vs 0.27%) and are mainly split by age and impression volume instead.

**Honest takeaway:** the clustering surfaces one clean rediscovery of the baseline's own pattern (Cluster 2 ≈ "stale but visible"), one genuinely new distinction the flat rule missed (Cluster 1's "buried regardless of age" pages vs. Cluster 4's "good position but no demand" pages — two very different problems the single-score baseline couldn't tell apart), and one clear failure mode (Cluster 3, a near-meaningless tiny outlier group). That mix — one confirmation, one new insight, one honest flaw — is a more useful finding than a clean score alone would be.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.